# FastHalluCheck: Qwen3-8B judge on Kaggle (2 x T4)

Runs `configs/judge_qwen8b.yaml`: the 20K/30K-token PHANTOM cells, 50 pairs per cell, with whole contexts in one 48K window. Resumable: if the 12-hour session ends, start a new one and run all cells again; finished cells are skipped as long as `results/` is restored from the saved output (step 4).

**Settings:** Accelerator = GPU T4 x2, Internet = on. Add a Kaggle secret `HF_TOKEN` (optional, faster downloads).

In [ ]:
!git clone -b claude/nifty-turing-2tmmro https://github.com/vaishnav-16/llm-hallucination-detection.git
%cd llm-hallucination-detection
!pip install -q -e ".[models]" accelerate

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('no HF_TOKEN secret:', e)
import torch; print(torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 1) data (PHANTOM long files only are needed, but prepare builds everything; ~5 min)
!python -m src.cli prepare --no-tokens

In [ ]:
# 2) optional: restore partial results from a previous session's output (attach it as a dataset first)
# !cp -r /kaggle/input/<previous-output>/results ./

In [ ]:
# 3) score
!python -m src.cli score --config configs/judge_qwen8b.yaml --device cuda

In [ ]:
# 4) save artifacts + partial shards to the notebook output (download, then commit artifacts/ from your laptop)
!mkdir -p /kaggle/working/out && cp -r artifacts/predictions /kaggle/working/out/ && cp -r results /kaggle/working/out/ 2>/dev/null; ls /kaggle/working/out/predictions | tail